# ChemPy Rust Integration Tutorial

This notebook demonstrates how the high-performance Rust implementation of ChemPy can be used directly from Python. We use **PyO3** to create bindings and **maturin** to build the extension.

## 1. Importing the Rust Module

First, we import the `chempy_rust` module, which is built from the Rust source in `src/`.

In [ ]:
import chempy_rust
print(f"ChemPy Rust module loaded: {chempy_rust}")

## 2. Working with Elements

We can access the periodic table data stored in Rust. Let's look up Carbon:

In [ ]:
c = chempy_rust.get_element(symbol="C")
print(f"Element: {c.name} ({c.symbol})")
print(f"Atomic Number: {c.number}")
print(f"Atomic Mass: {c.mass} kg/mol")

## 3. Thermodynamic Model Conversion

One of the high-performance features implemented in Rust is the ability to fit thermodynamic models to data. Here we fit a **Wilhoit model** to heat capacity data and then convert it to a **NASA polynomial model** using a high-speed linear solver (**nalgebra**).

In [ ]:
# 1. Create a Wilhoit model and fit it to Ethane data
wilhoit = chempy_rust.PyWilhoitModel(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 500.0)
t_data = [300.0, 400.0, 500.0, 600.0, 800.0, 1000.0, 1500.0]
cp_data = [52.4, 65.2, 77.8, 89.1, 107.5, 122.2, 146.4]
h298 = -84.0 * 1000.0
s298 = 229.5

wilhoit.fit_to_data(t_data, cp_data, linear=False, n_freq=18, n_rotors=1, h298=h298, s298=s298, b0=500.0)
print(f"Wilhoit Cp at 1000K: {wilhoit.get_heat_capacity(1000.0):.2f} J/mol*K")

# 2. Convert the Wilhoit model to a NASA polynomial model
nasa = chempy_rust.convert_wilhoit_to_nasa(wilhoit, t_min=300.0, t_max=3000.0, t_int=1000.0)
print(f"NASA Cp at 1000K: {nasa.get_heat_capacity(1000.0):.2f} J/mol*K")
print("Conversion complete using Rust linear algebra solver!")

## 4. Performance Comparison (Conceptual)

In a real-world scenario, you would use the Rust implementation for performance-critical tasks like:

- **Large-scale Graph Isomorphism:** Finding reaction sites in complex molecules.
- **High-speed ODE Integration:** Simulating chemical kinetics with thousands of species.
- **Thermodynamic Regressions:** Fitting NASA polynomials to experimental data.

The Rust implementation typically provides a 10-100x speedup over pure Python for these tasks.